# BayMorph — Assess an Observed Single-Band Image

This notebook performs the decisions that must precede inference. It reads a
user-supplied FITS image, lets the user identify the target visually, inspects
the image and FITS metadata, calibrates the observational support of the total
flux prior, and writes a reviewed BayMorph YAML configuration.

It does **not** run the production fit. Inference is performed by
`02B_Observed_Fit.ipynb` using the YAML created here.

Model screening and Bayesian model comparison are distinct. BayMorph does not
currently distribute a validated model-screening classifier, so this notebook
does not fabricate an image-based model ranking. The model below is an explicit
starting-model decision. A preferred model can be established only by fitting
the relevant alternatives and comparing their evidences and diagnostics.


In [ ]:
from pathlib import Path

from baymorph import ObservationAdvisor


## 1. Select the observation

Set the FITS path and analysis name. Relative paths are resolved automatically
from the BayMorph project root, even when Jupyter is running inside the
`notebooks` directory. Absolute FITS paths are also accepted.


In [ ]:
fits_path = Path("data/observed_galaxy.fits")
analysis_name = "observed_galaxy"
config_path = Path("configs") / f"{analysis_name}.yaml"

advisor = ObservationAdvisor.from_project(fits_path)


## 2. Identify the target

Click the target galaxy in the image. The cyan cross can also be positioned
using the numerical \(X,Y\) fields. Plotly's toolbar provides zoom and pan.

BayMorph uses zero-based image coordinates: \(X\) is the NumPy column and
\(Y\) is the NumPy row. Press **Confirm center** before continuing. A candidate
position is never accepted implicitly.


In [ ]:
target_selector = advisor.select_target(
    title="Observed image — select the target center"
)

target_selector


## 3. Inspect the FITS header and image

The Advisor reports the image dimensions, finite-pixel coverage, pixel scale,
seeing metadata, background and noise scales, and diagnostics for the selected
source.

Image-derived flux, size, axis ratio, and position angle are diagnostic
numerical scales. They are not silently converted into scientific prior bounds.
Review the target segmentation and any unavailable or ambiguous metadata before
continuing.


In [ ]:
inspection = advisor.inspect()

print(advisor.summary())


## 4. Declare the analysis

`single_sersic` is used here as the simplest baseline starting model. This is a
user decision, not a claim that the image is intrinsically single-Sérsic.

The likelihood is normalized because BayMorph reports Bayesian evidence. The
sampler settings below are a practical starting point and may be increased for
publication analyses.


In [ ]:
starting_model = "single_sersic"

# Define this from a predeclared sample- or instrument-level policy.
# Do not copy target-specific calibration bounds automatically.
# Use identical flux support for every model compared by evidence.
reviewed_flux_prior = None

# Expected schema after scientific review:
# reviewed_flux_prior = {
#     "type": "log_uniform",
#     "lower": ...,
#     "upper": ...,
# }

analysis_options = {
    "object_name": analysis_name,
    "outputs_root": "runs",
    "psf_type": "empirical",
    "likelihood": {
        "type": "gaussian",
        "include_normalization": True,
    },
    "sampler": {
        "name": "nautilus",
        "execution": {
            "processes": 1,
            "progress": True,
        },
        "init": {
            "n_live": 300,
            "seed": 42,
        },
        "run": {
            "n_eff": 500,
            "f_live": 0.02,
        },
    },
}


## 5. Calibrate the observational flux support

This stage prepares the image and empirical PSF, then evaluates the flux range
over which the declared morphology family is detectable without saturating the
detector.

The measured target flux does not define or centre the resulting bounds.
Inspect the calibration summary, especially any coverage warning. Posterior
contact with a prior boundary must still be checked after fitting.


In [ ]:
flux_calibration = advisor.calibrate_flux_prior(
    prepare=True,
    build_psf=True,
    snr_threshold=5.0,
    required_detectable_fraction=0.95,
    required_unsaturated_fraction=0.95,
    n_draws=1024,
    random_state=42,
    bootstrap_samples=512,
    confidence_level=0.95,
    model_name=starting_model,
    **analysis_options,
)

print(flux_calibration.summary())
print(
    "\nThe calibrated bounds above are operational diagnostics only; "
    "they are not inserted into the fitting configuration."
)


## 6. Review the proposed configuration

Creating the proposal does not write a file. It records the confirmed target,
the user-selected starting model, the origin of the flux prior, and any warning
that must be considered before interpreting Bayesian evidence.


In [ ]:
proposal = None

if reviewed_flux_prior is None:
    raise ValueError(
        "Define reviewed_flux_prior from a predeclared scientific "
        "policy before creating the fitting configuration."
    )

proposal = advisor.propose_config(
    output_path=config_path,
    model_name=starting_model,
    flux_prior=reviewed_flux_prior,
    **analysis_options,
)

proposal


## 7. Write the YAML

Run this cell only after reviewing the proposal. The explicit `accept=True` is
required; without it BayMorph refuses to write the configuration.


In [ ]:
if proposal is None:
    raise ValueError(
        "Create and review a valid configuration proposal before writing."
    )

config_path = proposal.write(
    accept=True
)

config_path
